# 🛡️ Model 06 — Support Vector Machine (SVM)

Using PCA (50 components) to handle high dimensionality and speed up training.

In [1]:
import os
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc

train = pd.read_csv('../data/train_balanced.csv').sample(min(100000, pd.read_csv('../data/train_balanced.csv', nrows=100001).shape[0]-1), random_state=42)
test = pd.read_csv('../data/test.csv')

X_train = train.drop(columns=['isFraud', 'TransactionID'], errors='ignore')
y_train = train['isFraud']
X_test = test.drop(columns=['isFraud', 'TransactionID'], errors='ignore')
y_test = test['isFraud']

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print('Reducing dimensions with PCA...')
pca = PCA(n_components=50)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print('Training Linear SVM...')
base_model = LinearSVC(dual=False, max_iter=2000)
model = CalibratedClassifierCV(base_model, cv=3)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
preds = model.predict(X_test)
print(classification_report(y_test, preds))
precision, recall, _ = precision_recall_curve(y_test, probs)
print(f'PR-AUC: {auc(recall, precision):.4f}')

Reducing dimensions with PCA...
Training Linear SVM...


              precision    recall  f1-score   support

           0       0.98      0.90      0.94    113975
           1       0.16      0.49      0.24      4133

    accuracy                           0.89    118108
   macro avg       0.57      0.70      0.59    118108
weighted avg       0.95      0.89      0.92    118108

PR-AUC: 0.2497
